# UNION and UNION ALL in Databricks PySpark

Union operations combine rows from multiple DataFrames into a single DataFrame. Understanding the differences between `union()`, `unionAll()`, and `unionByName()` is crucial for data merging operations.

## Core Functions

### 1. **union()**
Combines two DataFrames by appending rows. Does **NOT** remove duplicates.

**Syntax:**
```python
df1.union(df2)
```

**Behavior:**
* Combines DataFrames vertically (stacks rows)
* Does **NOT** remove duplicate rows
* Matches columns **by position**, not by name
* Both DataFrames must have the **same number of columns**
* Column names from the first DataFrame are used in the result
* **Equivalent to SQL UNION ALL** (despite the name)

### 2. **unionAll()** [DEPRECATED]
Legacy function that behaves identically to `union()`.

**Status:**
* **Deprecated since Spark 2.0**
* Maintained for backward compatibility
* **Use `union()` instead**
* Both functions do NOT remove duplicates

### 3. **unionByName()**
Combines DataFrames by matching **column names** instead of position.

**Syntax:**
```python
df1.unionByName(df2, allowMissingColumns=False)
```

**Behavior:**
* Matches columns by name, not position
* Column order doesn't matter
* `allowMissingColumns=True`: Fills missing columns with NULL
* `allowMissingColumns=False` (default): Raises error if columns don't match
* **Safer than union()** when column order may vary

## Key Differences

| Feature                | union()              | unionByName()           | SQL UNION        | SQL UNION ALL    |
| ---------------------- | -------------------- | ----------------------- | ---------------- | ---------------- |
| Removes Duplicates     | **No**               | **No**                  | **Yes**          | **No**           |
| Matches Columns By     | Position             | Name                    | Position         | Position         |
| Column Order Matters   | Yes                  | No                      | Yes              | Yes              |
| Performance            | Fast                 | Fast                    | Slower (dedup)   | Fast             |
| PySpark Equivalent     | `union()`            | `unionByName()`         | `union().distinct()` | `union()`    |

## Important Concept: union() = SQL UNION ALL

**Common Confusion:**
* PySpark `union()` does **NOT** remove duplicates
* SQL `UNION` **DOES** remove duplicates
* PySpark `union()` = SQL `UNION ALL`

**To get SQL UNION behavior in PySpark:**
```python
# SQL: SELECT * FROM table1 UNION SELECT * FROM table2
# PySpark equivalent:
df1.union(df2).distinct()
```

## Examples

### Example 1: Basic Union (Positional Matching)
```python
from pyspark.sql import Row

# Create sample DataFrames
df1 = spark.createDataFrame([
    Row(id=1, name="Alice", age=25),
    Row(id=2, name="Bob", age=30)
])

df2 = spark.createDataFrame([
    Row(id=3, name="Charlie", age=35),
    Row(id=2, name="Bob", age=30)  # Duplicate
])

# Union - keeps duplicates
result = df1.union(df2)
# Result has 4 rows (duplicate Bob row is kept)

# Union with deduplication (SQL UNION equivalent)
result_distinct = df1.union(df2).distinct()
# Result has 3 rows (duplicate removed)
```

### Example 2: Union with Different Column Order
```python
# df1 columns: id, name, age
df1 = spark.createDataFrame([
    Row(id=1, name="Alice", age=25)
])

# df2 columns: id, age, name (different order!)
df2 = spark.createDataFrame([
    Row(id=2, age=30, name="Bob")
])

# union() matches by POSITION - WRONG RESULT!
result_wrong = df1.union(df2)
# Column 2 of df2 (age=30) goes into 'name' column!

# unionByName() matches by NAME - CORRECT RESULT
result_correct = df1.unionByName(df2)
# Columns matched correctly by name
```

### Example 3: Union with Missing Columns
```python
df1 = spark.createDataFrame([
    Row(id=1, name="Alice", age=25, city="NYC")
])

df2 = spark.createDataFrame([
    Row(id=2, name="Bob", age=30)  # Missing 'city' column
])

# This will fail - column count mismatch
# df1.union(df2)  # ERROR!

# unionByName with allowMissingColumns - fills with NULL
result = df1.unionByName(df2, allowMissingColumns=True)
# Bob's row will have city=NULL
```

### Example 4: Multiple DataFrame Union
```python
from functools import reduce

df_list = [df1, df2, df3, df4]

# Union multiple DataFrames
result = reduce(lambda x, y: x.union(y), df_list)

# Or using unionByName
result = reduce(lambda x, y: x.unionByName(y), df_list)
```

### Example 5: Union Different Data Types
```python
df1 = spark.createDataFrame([
    Row(id=1, value="100")  # value is string
])

df2 = spark.createDataFrame([
    Row(id=2, value=200)  # value is integer
])

# Union works but type is widened/cast
result = df1.union(df2)
# Result: value column will be string type (wider type)
```

## SQL Usage

### SQL UNION (Removes Duplicates)
```sql
SELECT id, name, age FROM table1
UNION
SELECT id, name, age FROM table2;
-- Removes duplicate rows
```

### SQL UNION ALL (Keeps Duplicates)
```sql
SELECT id, name, age FROM table1
UNION ALL
SELECT id, name, age FROM table2;
-- Keeps all rows including duplicates
-- Equivalent to PySpark union()
```

## Best Practices

### 1. **Prefer unionByName() for Safety**
```python
# Safer - column names must match
df1.unionByName(df2)

# Risky - columns matched by position only
df1.union(df2)
```

### 2. **Check Schemas Before Union**
```python
# Verify schemas match
print("DF1 Schema:", df1.schema)
print("DF2 Schema:", df2.schema)

# Or compare programmatically
if df1.schema == df2.schema:
    result = df1.union(df2)
else:
    print("Schema mismatch!")
```

### 3. **Handle Schema Differences**
```python
# Add missing columns before union
if "city" not in df2.columns:
    from pyspark.sql.functions import lit
    df2 = df2.withColumn("city", lit(None))

result = df1.union(df2)
```

### 4. **Remove Duplicates When Needed**
```python
# If you want SQL UNION behavior
result = df1.union(df2).distinct()

# Or use dropDuplicates for specific columns
result = df1.union(df2).dropDuplicates(["id"])
```

## Performance Considerations

### Fast Operations (No Shuffle)
* `union()` - Just appends partitions
* `unionAll()` - Same as union()
* `unionByName()` - Column name matching is metadata operation

### Slow Operations (Requires Shuffle)
* `distinct()` - Full shuffle to find unique rows
* `dropDuplicates()` - Shuffle on specified columns

### Optimization Tips
```python
# BAD: Multiple unions then distinct
result = df1.union(df2).union(df3).union(df4).distinct()
# Problem: Large intermediate DataFrame before deduplication

# BETTER: Remove duplicates from individual DataFrames first
df1_clean = df1.distinct()
df2_clean = df2.distinct()
df3_clean = df3.distinct()
df4_clean = df4.distinct()
result = df1_clean.union(df2_clean).union(df3_clean).union(df4_clean).distinct()
```

## Common Pitfalls

### ❌ **Pitfall 1: Assuming union() Removes Duplicates**
```python
# WRONG: Expecting unique rows
result = df1.union(df2)  # Keeps duplicates!

# CORRECT: Explicitly remove duplicates
result = df1.union(df2).distinct()
```

### ❌ **Pitfall 2: Column Order Mismatch**
```python
# df1: id, name, age
# df2: id, age, name  (different order)

# WRONG: Values go into wrong columns
result = df1.union(df2)

# CORRECT: Use unionByName
result = df1.unionByName(df2)
```

### ❌ **Pitfall 3: Data Type Mismatch**
```python
# df1: id (integer), name (string)
# df2: id (string), name (string)

# Union works but types may be widened/cast
result = df1.union(df2)
# Better: Cast explicitly before union
df2 = df2.withColumn("id", col("id").cast("integer"))
result = df1.union(df2)
```

## Use Cases

* **Combining partitioned data**: Merge data from multiple time periods or regions
* **Incremental loads**: Append new data to existing datasets
* **Merging similar tables**: Combine data from different sources with same schema
* **Historical data**: Union current and archived data for analysis
* **A/B testing**: Combine control and treatment group data
* **Multi-source aggregation**: Combine logs or events from multiple systems

### Create dataframe DF1

In [0]:
employee_data= [
  (100, "Scott", 5000, "Sales", "Male"),
  (200, "Neil", 6000, "Sales", "Male"),
  (300, "Amy", 7000, "Finance", "Female"),
  (400, "Karen", 8000, "Finance", "Female"),
]

schema = 'id INT, name STRING, salary INT, dept STRING, gender STRING'

df1 = spark.createDataFrame(employee_data, schema)

df1.show()

###  Create dataframe DF2

In [0]:
employee_data= [
  (100, "Scott", 5000, "Sales", "Male"),
  (300, "Amy", 7000, "Finance", "Female"),
  (500, "Ravi", 9000, "Finance", "Male") ,
  (600, "Raj", 10000, "Finance", "Male")
]

schema = 'id INT, name STRING, salary INT, dept STRING, gender STRING'

df2 = spark.createDataFrame(employee_data, schema)

df2.show()

In [0]:
# UNION
print('UNION')
df1.union(df2).show()

# UNIONALL
print('UNIONALL')
df1.unionAll(df2).show()

# INTERSECT
print('INTERSECT')
df1.intersect(df2).show()

# EXCEPT
print('EXCEPT')
df1.exceptAll(df2).show()

### Drop Duplicates

In [0]:
df1.union(df2).dropDuplicates().show()

In [0]:
df1.union(df2).dropDuplicates(['id']).show()

In [0]:
df1.union(df2).dropDuplicates(['dept']).show()

### Create dataframe DF3

In [0]:
employee_data= [
  (800, "raj", "Male", "Sales", 5000),
  (900, "ravi", "Male", "Sales", 6000),
  (1000, "karen", "Female", "Sales", 7000),
  (1100, "amy", "Female", "Sales", 8000)
]

schema = 'id INT, name STRING, gender STRING, dept STRING, salary INT'

df3 = spark.createDataFrame(employee_data, schema)

df3.show()

In [0]:
df2.unionByName(df3).show()

In [0]:
employee_data= [
  (800, "raj", "Sales"),
  (900, "ravi", "Sales")
]

schema = 'id INT, name STRING, dept STRING'

df4 = spark.createDataFrame(employee_data, schema)

df4.show()

In [0]:
df1.unionByName(df4, allowMissingColumns=True).show()

### Multiple data frames

In [0]:
df1.unionByName(df2, allowMissingColumns=True).unionByName(df3, allowMissingColumns=True).unionByName(df4, allowMissingColumns=True).dropDuplicates(['id']).show()